# Table V on Colab: losses, sentence head, distillation

Runs the three experiments of Table V with `notebooks/09_pieces_234.py`, on
validation, with the settings reported in the paper:

| Experiment | Command | Seeds |
|---|---|---|
| balanced losses | `--piece3` | 3 |
| sentence head | `--piece2` | 5 |
| distillation from SemiSOLD | `--piece4 --lambda 0.3 --exclude-sold` | 5 |

Distillation removes every SemiSOLD tweet that is also a SOLD tweet before
training (see `src/semisold.py`).

1. Runtime -> Change runtime type -> T4 GPU
2. Run all cells. Results are also copied to Google Drive, so a disconnect does
   not lose them. Expect several hours in total.

## 1. GPU

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU'

## 2. Code and dependencies

In [ ]:
import os, shutil
REPO = 'https://github.com/hatheem-r/project_DNN.git'
if os.path.exists('/content/project'):
    shutil.rmtree('/content/project')
%cd /content
!git clone -q $REPO project
%cd /content/project
!pip install -q 'datasets<3.0.0' pytorch-crf sentencepiece 2>&1 | tail -1

## 3. Tests

In [ ]:
!python tests/test_metrics.py | tail -2
!python tests/test_subword_alignment.py | tail -2

## 4. Google Drive for the outputs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/sold_results'
os.makedirs(OUT, exist_ok=True)
print('saving to', OUT)

## 5. fastText vectors (about 460 MB)

In [ ]:
!mkdir -p embeddings results artifacts
![ -f embeddings/cc.si.300.vec.gz ] || wget -q -O embeddings/cc.si.300.vec.gz https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.si.300.vec.gz
!python notebooks/03_embeddings.py | tail -3

## 6. Balanced losses

In [ ]:
!python notebooks/09_pieces_234.py --piece3 2>&1 | tee results/piece3_report.txt | tail -20
!cp results/piece3_report.txt results/results_piece3.csv $OUT/

## 7. Sentence head

In [ ]:
!python notebooks/09_pieces_234.py --piece2 --seeds 1 2 3 4 5 2>&1 | tee results/piece2_report.txt | tail -20
!cp results/piece2_report.txt results/results_piece2.csv $OUT/

## 8. Distillation without SOLD tweets in SemiSOLD

In [ ]:
!python notebooks/09_pieces_234.py --piece4 --lambda 0.3 --exclude-sold --seeds 1 2 3 4 5 2>&1 | tee results/piece4_nosold_report.txt | tail -20
!cp results/piece4_nosold_report.txt results/results_piece4_nosold.csv $OUT/
!ls -la $OUT